# XGBoost Classifier — MSME Project Completion Status Prediction
### DOST SETUP 4.0 iFund Program, Western Visayas
**Student:** Shi | **Model:** XGBoost (Extreme Gradient Boosting)

**Target Variable:** `Completion_Status` (Completed = 1, Not Completed = 0)

**Dataset:** `MSME_data_cleaned.xlsx` — 321 records, 6 provinces including Aklan

---
### Process Flow (per Adviser Sir Paolo)
```
Raw Data (321 records)
  → Drop leakage columns FIRST (Refund_Status, Beneficiary_Name, Year)
  → Train-Test Split 80/20 stratified — BEFORE any cleaning
  → Normalize & Encode TRAIN and TEST separately — no leakage
  → Feature Engineering — separately on train and test
  → SMOTE on TRAIN only — fix class imbalance
  → RandomizedSearchCV — auto hyperparameter tuning (optimize AUC)
  → Final evaluation on TEST SET only (AUC, F1, Confusion Matrix)
  → Save model as xgb_final.pkl
```

**Key decisions per Sir Paolo's guidelines:**
- `Refund_Status` dropped before splitting — directly encodes outcome = **data leakage**
- Split **FIRST**, clean after — never mix train and test cleaning
- Encoders fit on training set only, applied to test separately
- SMOTE on **training set only** — never on test
- Hyperparameters **auto-tuned** via RandomizedSearchCV — not fixed manually
- Primary metrics: **AUC and F1-score** — not accuracy
- Confusion matrix from **test set predictions only**

## 0. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection  import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing    import LabelEncoder
from sklearn.metrics          import (f1_score, roc_auc_score, confusion_matrix,
                                      classification_report, ConfusionMatrixDisplay,
                                      roc_curve, precision_recall_curve)
from imblearn.over_sampling   import SMOTE
from xgboost                  import XGBClassifier
from scipy.stats              import randint, uniform

RANDOM_STATE = 42
print("Libraries loaded successfully.")

## 1. Load Raw Dataset

In [ ]:
df_raw = pd.read_excel('MSME_data_cleaned.xlsx')

print(f"Shape: {df_raw.shape}")
print(f"\nColumns: {list(df_raw.columns)}")
print(f"\nTarget variable distribution:")
vc  = df_raw['Completion_Status'].value_counts()
pct = df_raw['Completion_Status'].value_counts(normalize=True).mul(100).round(1)
for label in vc.index:
    print(f"  {label:<20} {vc[label]:>4}  ({pct[label]}%)")
df_raw.head()

## 2. Drop Leakage and Non-Predictive Columns

**Dropped BEFORE any splitting:**
- `Refund_Status` — directly encodes outcome (Graduated, Terminated...) = **DATA LEAKAGE**
- `Beneficiary_Name` — identifier only, not a predictive feature
- `Year` — encodes time bias; won't generalize to future applicants

In [ ]:
DROP_COLS = ['Beneficiary_Name', 'Year', 'Refund_Status']
df = df_raw.drop(columns=DROP_COLS).copy()

print(f"Dropped  : {DROP_COLS}")
print(f"Remaining: {list(df.columns)}")
print(f"Shape    : {df.shape}")

## 3. Define Features and Target

In [ ]:
FEATURES = ['Province', 'Sector', 'type_of_ownership',
            'size_of_enterprise', 'Project_Cost', 'Has_Prior_Funding']
TARGET = 'Completion_Status'

X = df[FEATURES].copy()
y = (df[TARGET] == 'Completed').astype(int)  # 1=Completed, 0=Not Completed

print(f"Features : {FEATURES}")
print(f"Target   : 1=Completed | 0=Not Completed")
print(f"\nClass distribution:")
print(pd.Series(y).value_counts().rename({1:'Completed',0:'Not Completed'}))

## 4. Normalize Sector Spelling Variants

Consolidates misspelled and variant sector labels before splitting.
This is a label correction — not imputation, so it does not introduce leakage.

In [ ]:
sector_map = {
    'Agriculture/Marine/Aquaculture' : 'Horticulture & Agriculture',
    'Horticulture and Agriculture'   : 'Horticulture & Agriculture',
}
X['Sector'] = X['Sector'].replace(sector_map)
X['Sector'] = X['Sector'].where(~X['Sector'].str.startswith('Others', na=False),
                                 'Others (grouped)')
print("Sector values after normalization:")
print(X['Sector'].value_counts())

## 5. Train-Test Split ← FIRST Before Any Cleaning

**Per Sir Paolo:** Split the data FIRST, then clean each set separately.
Stratified split ensures both sets preserve the original class ratio.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    random_state = RANDOM_STATE,
    stratify     = y
)

print(f"Training rows : {X_train.shape[0]}")
print(f"Test rows     : {X_test.shape[0]}")
print(f"\nTrain class distribution:")
print(pd.Series(y_train).value_counts().rename({1:'Completed',0:'Not Completed'}))
print(f"\nTest class distribution:")
print(pd.Series(y_test).value_counts().rename({1:'Completed',0:'Not Completed'}))

## 6. Encode Categoricals — Train and Test Separately

`LabelEncoder` is **fit on TRAIN only** then applied to TEST.
This ensures no test information leaks into the training process.

In [ ]:
CAT_COLS = ['Province', 'Sector', 'type_of_ownership', 'size_of_enterprise']

encoders    = {}
X_train_enc = X_train.copy()
X_test_enc  = X_test.copy()

for col in CAT_COLS:
    le = LabelEncoder()
    X_train_enc[col] = le.fit_transform(X_train[col].astype(str))    # fit on TRAIN
    X_test_enc[col]  = X_test[col].astype(str).map(                  # apply to TEST
        lambda x, le=le: le.transform([x])[0] if x in le.classes_ else -1
    )
    encoders[col] = le
    print(f"  '{col}': {list(le.classes_)}")

X_train_enc['Has_Prior_Funding'] = X_train_enc['Has_Prior_Funding'].astype(int)
X_test_enc['Has_Prior_Funding']  = X_test_enc['Has_Prior_Funding'].astype(int)
print("\nEncoding complete.")

## 7. Feature Engineering — Separately on Train and Test

Two new features created to improve predictive power:
- `Cost_to_Size_Ratio` — project cost relative to enterprise size
- `Log_Project_Cost` — log-transformed cost (reduces right skewness)

Engineered **separately** on train and test — no leakage.

In [ ]:
size_num_map = {'micro': 1, 'small': 2, 'medium': 3}

def add_features(X_raw, X_enc):
    X_enc    = X_enc.copy()
    size_num = X_raw['size_of_enterprise'].str.lower().map(size_num_map)
    X_enc['Cost_to_Size_Ratio'] = X_raw['Project_Cost'].values / size_num.values
    X_enc['Log_Project_Cost']   = np.log1p(X_raw['Project_Cost'].values)
    return X_enc

X_train_enc = add_features(X_train, X_train_enc)
X_test_enc  = add_features(X_test,  X_test_enc)

print(f"Final feature set ({len(X_train_enc.columns)} features):")
for f in X_train_enc.columns:
    print(f"  {f}")
print(f"\nTrain shape: {X_train_enc.shape} | Test shape: {X_test_enc.shape}")

## 8. SMOTE — Training Set Only

**SMOTE** (Synthetic Minority Oversampling Technique) generates synthetic samples
for the minority class (Not Completed) to fix class imbalance.

Applied to **training set only** — never on test set.
Applying SMOTE to test data would give unrealistic evaluation results.

In [ ]:
print("Before SMOTE (train):")
print(pd.Series(y_train).value_counts().rename({1:'Completed',0:'Not Completed'}))

smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train_enc, y_train)

print("\nAfter SMOTE (train):")
print(pd.Series(y_train_sm).value_counts().rename({1:'Completed',0:'Not Completed'}))
print(f"\nTraining rows after SMOTE: {X_train_sm.shape[0]}")

## 9. Hyperparameter Tuning — RandomizedSearchCV

**Per Sir Paolo:** Hyperparameters must NOT be fixed manually.
`RandomizedSearchCV` with 80 iterations automatically finds the best configuration.

Primary tuning metric: **ROC-AUC** — not accuracy.

In [ ]:
param_dist = {
    'n_estimators'    : randint(200, 600),
    'max_depth'       : randint(3, 8),
    'learning_rate'   : uniform(0.01, 0.2),
    'subsample'       : uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_weight': randint(1, 6),
    'gamma'           : uniform(0, 0.3),
    'reg_alpha'       : uniform(0, 0.5),
    'reg_lambda'      : uniform(0.5, 2.0),
}

xgb_base = XGBClassifier(
    objective        = 'binary:logistic',
    eval_metric      = 'auc',
    use_label_encoder= False,
    tree_method      = 'hist',
    random_state     = RANDOM_STATE
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rs = RandomizedSearchCV(
    estimator           = xgb_base,
    param_distributions = param_dist,
    n_iter              = 80,
    scoring             = 'roc_auc',
    cv                  = cv,
    n_jobs              = -1,
    verbose             = 1,
    random_state        = RANDOM_STATE
)

print("Running RandomizedSearchCV (80 iterations x 5-fold CV)...")
rs.fit(X_train_sm, y_train_sm)

print(f"\nBest parameters (auto-selected):")
for k, v in rs.best_params_.items():
    print(f"  {k:<22}: {round(float(v), 4)}")
print(f"\nBest Cross-Validation AUC: {rs.best_score_:.4f}")

### 9.1 Cross-Validation — Top 5 Configurations

In [ ]:
cv_res = pd.DataFrame(rs.cv_results_)
top5   = (cv_res[['params','mean_test_score','std_test_score','rank_test_score']]
          .sort_values('rank_test_score').head(5).reset_index(drop=True))
top5.columns = ['Parameters','Mean AUC','Std AUC','Rank']
top5['Mean AUC'] = top5['Mean AUC'].round(4)
top5['Std AUC']  = top5['Std AUC'].round(4)
print("=== Top 5 Configurations by Cross-Validation AUC ===")
top5[['Rank','Mean AUC','Std AUC']]

## 10. Final Evaluation on Test Set

**Per Sir Paolo:** Confusion matrix and all metrics must be evaluated on the **test set only**.
The best model from RandomizedSearchCV is used directly.

In [ ]:
best_model   = rs.best_estimator_
y_pred       = best_model.predict(X_test_enc)
y_pred_proba = best_model.predict_proba(X_test_enc)[:, 1]

test_auc = roc_auc_score(y_test, y_pred_proba)
f1_w     = f1_score(y_test, y_pred, average='weighted')
f1_nc    = f1_score(y_test, y_pred, pos_label=0)
f1_c     = f1_score(y_test, y_pred, pos_label=1)

print("=== Test Set Performance ===")
print(f"ROC-AUC Score          : {test_auc:.4f}")
print(f"Weighted F1-Score      : {f1_w:.4f}")
print(f"F1 (Not Completed)     : {f1_nc:.4f}")
print(f"F1 (Completed)         : {f1_c:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred,
      target_names=['Not Completed','Completed']))

### 10.1 Confusion Matrix (Test Set Only)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['Not Completed','Completed']).plot(
    ax=ax, cmap='Blues', colorbar=False)
ax.set_title('XGBoost — Confusion Matrix (Test Set)', fontsize=12)
plt.tight_layout()
plt.savefig('xgb_confusion_matrix.png', dpi=150)
plt.show()

print(f"True Negatives  (Correctly predicted Not Completed) : {tn}")
print(f"False Positives (Predicted Completed, was Not)      : {fp}")
print(f"False Negatives (Predicted Not Completed, was done) : {fn}")
print(f"True Positives  (Correctly predicted Completed)     : {tp}")

### 10.2 ROC Curve (Test Set)

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'XGBoost (AUC = {test_auc:.3f})')
ax.plot([0,1],[0,1], color='gray', linestyle='--', lw=1, label='Random Chance')
ax.fill_between(fpr, tpr, alpha=0.1, color='steelblue')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — XGBoost (Test Set)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('xgb_roc_curve.png', dpi=150)
plt.show()

### 10.3 Precision-Recall Curve (Test Set)

In [ ]:
prec, rec, _ = precision_recall_curve(y_test, y_pred_proba)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(rec, prec, color='darkorange', lw=2)
ax.fill_between(rec, prec, alpha=0.1, color='darkorange')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve — XGBoost (Test Set)')
plt.tight_layout()
plt.savefig('xgb_precision_recall.png', dpi=150)
plt.show()

## 11. Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'Feature'   : X_train_enc.columns,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print("=== Feature Importance ===")
print(importance_df.to_string(index=False))

import matplotlib.patches as mpatches
fig, ax = plt.subplots(figsize=(9, 5))
colors  = ['#1565C0' if i < 3 else '#42A5F5'
           for i in range(len(importance_df)-1, -1, -1)]
bars    = ax.barh(importance_df['Feature'][::-1],
                  importance_df['Importance'][::-1], color=colors)
for bar, val in zip(bars, importance_df['Importance'][::-1]):
    ax.text(bar.get_width()+0.002, bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
ax.set_xlabel('Importance Score')
ax.set_title('XGBoost Feature Importance', fontsize=13)
top3   = mpatches.Patch(color='#1565C0', label='Top 3 features')
others = mpatches.Patch(color='#42A5F5', label='Other features')
ax.legend(handles=[top3, others], loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('xgb_feature_importance.png', dpi=150)
plt.show()

## 12. Save Model as Pickle (for Streamlit Deployment)

**What is a .pkl file?**
Think of it like putting a cooked meal in the fridge — the Python script does the cooking
(training), and Streamlit just reheats it (loads it) without retraining from scratch.

Per Sir Paolo: *"A pickle file is needed. The input data from the five explanatory variables
should pass through a pipeline that converts the inputs into a data frame."*

In [ ]:
with open('xgb_final.pkl', 'wb') as f:
    pickle.dump({
        'model'        : best_model,
        'encoders'     : encoders,
        'sector_map'   : sector_map,
        'size_num_map' : size_num_map,
        'feature_cols' : list(X_train_enc.columns),
        'best_params'  : rs.best_params_,
        'cv_auc'       : rs.best_score_,
        'test_auc'     : test_auc,
        'test_f1'      : f1_w,
        'f1_nc'        : f1_nc,
        'f1_c'         : f1_c,
        'importance_df': importance_df,
    }, f)

print("Saved: xgb_final.pkl")
print("  Load this in: SHI_XGBoost_Streamlit_Final.py")

## 13. Final Summary

In [ ]:
print("=" * 65)
print("FINAL SUMMARY — XGBoost MSME Completion Predictor")
print("=" * 65)
print(f"Dataset         : {df.shape[0]} records | 6 provinces (Aklan included)")
print(f"Class balance   : 223 Completed / 98 Not Completed")
print(f"Train / Test    : {X_train.shape[0]} / {X_test.shape[0]} rows (80/20 stratified)")
print(f"Imbalance fix   : SMOTE on training set only")
print(f"Tuning method   : RandomizedSearchCV 80 iterations, 5-fold StratifiedKFold")
print(f"Tuning metric   : ROC-AUC (not accuracy, per Sir Paolo)")
print(f"\nBest Hyperparameters (auto-selected):")
for k, v in rs.best_params_.items():
    print(f"  {k:<22}: {round(float(v), 4)}")
print(f"\nModel Performance:")
print(f"  Cross-Validation AUC : {rs.best_score_:.4f}")
print(f"  Test AUC             : {test_auc:.4f}")
print(f"  Weighted F1-Score    : {f1_w:.4f}")
print(f"  F1 (Not Completed)   : {f1_nc:.4f}")
print(f"  F1 (Completed)       : {f1_c:.4f}")
print(f"\nTop 3 Most Important Features:")
for _, row in importance_df.head(3).iterrows():
    print(f"  {row['Feature']:<24}: {row['Importance']:.4f}")
print("=" * 65)